# Sistema de Conteo Automático de Vehículos con YOLO
**Práctica: UT03-3.7 — Visión Artificial y YOLO para Video**

Este notebook detecta y cuenta vehículos (coches, motos, autobuses y camiones) en un vídeo de tráfico usando YOLOv8 y OpenCV.

## 1. Instalación de dependencias

In [8]:
# Instalar las librerías necesarias (solo si no están instaladas)
# %pip install ultralytics opencv-python

## 2. Importaciones

In [9]:
import cv2
from ultralytics import YOLO
from collections import defaultdict

## 3. Configuración general

| Clase COCO | ID | Descripción |
|---|---|---|
| car | 2 | Coche |
| motorcycle | 3 | Moto |
| bus | 5 | Autobús |
| truck | 7 | Camión |

In [10]:
# ─── Parámetros ajustables ─────────────────────────────────────────
VIDEO_ENTRADA  = "video_trafico.mp4"   # Ruta al vídeo de entrada
VIDEO_SALIDA   = "trafficCam.mp4"      # Ruta al vídeo de salida anotado
MODELO         = "yolo11n.pt"          # Modelo YOLO (nano = más rápido)
CONFIANZA      = 0.25                  # Umbral de confianza (0-1)
CLASES_VEHICULOS = [2, 3, 5, 7]       # car, motorcycle, bus, truck

# Nombres legibles para mostrar en los conteos
NOMBRES_CLASES = {
    2: "Coche",
    3: "Moto",
    5: "Autobús",
    7: "Camión"
}

## 4. Prueba rápida: detección en un solo fotograma

Antes de procesar el vídeo completo, verificamos que el modelo carga correctamente y detecta vehículos.

In [11]:
# Cargar el modelo (se descarga automáticamente si no existe)
model = YOLO(MODELO)
print(f"Modelo '{MODELO}' cargado correctamente.")

# Leer el primer fotograma del vídeo para comprobar
cap_test = cv2.VideoCapture(VIDEO_ENTRADA)
ret, frame_test = cap_test.read()
cap_test.release()

if not ret:
    print("ERROR: No se pudo abrir el vídeo. Comprueba la ruta:", VIDEO_ENTRADA)
else:
    results_test = model(frame_test, conf=CONFIANZA, classes=CLASES_VEHICULOS)
    detecciones = len(results_test[0].boxes)
    print(f"Prueba OK — Vehículos detectados en el primer fotograma: {detecciones}")

Modelo 'yolo11n.pt' cargado correctamente.

0: 384x640 7 cars, 2.5ms
Speed: 0.8ms preprocess, 2.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Prueba OK — Vehículos detectados en el primer fotograma: 7


## 5. Procesamiento completo del vídeo con conteo de vehículos

Lógica de conteo:
- Se define una **línea virtual horizontal** a mitad del fotograma.
- Cuando el **centro** de una caja detectada cruza esa línea, el vehículo se registra.
- Se usa el **ID de seguimiento** (tracking) de YOLO para evitar contar el mismo vehículo varias veces.

In [12]:
# ─── Cargar modelo ─────────────────────────────────────────────────
model = YOLO(MODELO)

# ─── Abrir vídeo de entrada ─────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_ENTRADA)

if not cap.isOpened():
    raise FileNotFoundError(f"No se puede abrir el vídeo: {VIDEO_ENTRADA}")

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps   = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Vídeo: {ancho}x{alto} px | {fps} FPS | {total_frames} fotogramas")

# ─── Configurar escritor de vídeo de salida ─────────────────────────
writer = cv2.VideoWriter(
    VIDEO_SALIDA,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (ancho, alto)
)

# ─── Variables de conteo ─────────────────────────────────────────────
LINEA_Y = alto // 2              # Línea virtual horizontal a mitad del fotograma
IDS_CONTADOS = set()             # IDs de tracking ya contados (evita duplicados)
conteo_por_clase = defaultdict(int)  # {clase_id: cantidad}
frame_num = 0

print("\nProcesando vídeo...")

# ─── Bucle principal frame a frame ──────────────────────────────────
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break  # Fin del vídeo

    frame_num += 1

    # ── Detección + Tracking con YOLO ──────────────────────────────
    # persist=True activa el seguimiento entre fotogramas (ByteTrack)
    results = model.track(
        frame,
        conf=CONFIANZA,
        classes=CLASES_VEHICULOS,
        persist=True,
        verbose=False
    )

    # ── Dibujar anotaciones YOLO en el fotograma ───────────────────
    frame_anotado = results[0].plot()

    # ── Lógica de conteo por línea virtual ─────────────────────────
    if results[0].boxes is not None and results[0].boxes.id is not None:
        cajas    = results[0].boxes.xyxy.cpu().numpy()    # [x1, y1, x2, y2]
        track_ids = results[0].boxes.id.int().cpu().numpy()  # ID único por objeto
        clases   = results[0].boxes.cls.int().cpu().numpy()  # Clase (2,3,5,7)

        for caja, tid, cls_id in zip(cajas, track_ids, clases):
            x1, y1, x2, y2 = caja
            centro_y = int((y1 + y2) / 2)

            # Si el centro del objeto cruza la línea y no fue contado aún
            if centro_y > LINEA_Y and tid not in IDS_CONTADOS:
                IDS_CONTADOS.add(tid)
                conteo_por_clase[int(cls_id)] += 1

    # ── Dibujar línea virtual de conteo ───────────────────────────
    cv2.line(frame_anotado, (0, LINEA_Y), (ancho, LINEA_Y), (0, 255, 255), 2)
    cv2.putText(frame_anotado, "Linea de conteo",
                (10, LINEA_Y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # ── Mostrar totales acumulados en el fotograma ─────────────────
    total_general = sum(conteo_por_clase.values())
    y_texto = 30
    cv2.putText(frame_anotado, f"TOTAL: {total_general}",
                (10, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    for cls_id, cantidad in sorted(conteo_por_clase.items()):
        y_texto += 28
        nombre = NOMBRES_CLASES.get(cls_id, str(cls_id))
        cv2.putText(frame_anotado, f"  {nombre}: {cantidad}",
                    (10, y_texto), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 255, 100), 2)

    # ── Guardar fotograma en el vídeo de salida ────────────────────
    writer.write(frame_anotado)

    # ── Mostrar progreso en consola cada 50 frames ─────────────────
    if frame_num % 50 == 0:
        print(f"  Fotograma {frame_num}/{total_frames} — Vehículos contados: {total_general}")

# ─── Liberar recursos ────────────────────────────────────────────────
cap.release()
writer.release()
# cv2.destroyAllWindows()

print("\n✅ Procesamiento completado.")
print(f"Vídeo guardado en: {VIDEO_SALIDA}")

Vídeo: 1280x720 px | 30 FPS | 9184 fotogramas

Procesando vídeo...
  Fotograma 50/9184 — Vehículos contados: 6
  Fotograma 100/9184 — Vehículos contados: 8
  Fotograma 150/9184 — Vehículos contados: 11
  Fotograma 200/9184 — Vehículos contados: 12
  Fotograma 250/9184 — Vehículos contados: 15
  Fotograma 300/9184 — Vehículos contados: 21
  Fotograma 350/9184 — Vehículos contados: 25
  Fotograma 400/9184 — Vehículos contados: 29
  Fotograma 450/9184 — Vehículos contados: 31
  Fotograma 500/9184 — Vehículos contados: 32
  Fotograma 550/9184 — Vehículos contados: 34
  Fotograma 600/9184 — Vehículos contados: 36
  Fotograma 650/9184 — Vehículos contados: 37
  Fotograma 700/9184 — Vehículos contados: 39
  Fotograma 750/9184 — Vehículos contados: 41
  Fotograma 800/9184 — Vehículos contados: 43
  Fotograma 850/9184 — Vehículos contados: 49
  Fotograma 900/9184 — Vehículos contados: 54
  Fotograma 950/9184 — Vehículos contados: 57
  Fotograma 1000/9184 — Vehículos contados: 58
  Fotograma 105

## 6. Resultados finales

In [14]:
print("=" * 40)
print("   CONTEO FINAL DE VEHÍCULOS")
print("=" * 40)

for cls_id in CLASES_VEHICULOS:
    nombre   = NOMBRES_CLASES[cls_id]
    cantidad = conteo_por_clase.get(cls_id, 0)
    print(f"  {nombre:<12}: {cantidad:>4}")

print("-" * 40)
print(f"  {'TOTAL':<12}: {sum(conteo_por_clase.values()):>4}")
print("=" * 40)

   CONTEO FINAL DE VEHÍCULOS
  Coche       :  364
  Moto        :    2
  Autobús     :   53
  Camión      :  143
----------------------------------------
  TOTAL       :  562


---

## 7. Fuentes y Herramientas

### Documentación oficial consultada

- **Ultralytics YOLO Docs** — `https://docs.ultralytics.com`
  - Sección *Predict* para entender los parámetros `conf`, `classes` y `persist`.
  - Sección *Track* para el uso de `model.track()` con ByteTrack y el acceso a `boxes.id`.
- **OpenCV Docs** — `https://docs.opencv.org`
  - `cv2.VideoCapture` y `cv2.VideoWriter` para abrir y guardar vídeos.
  - `cv2.putText` y `cv2.line` para dibujar texto y líneas sobre los fotogramas.

### Sobre el código facilitado

El código base del enunciado proporciona la estructura del bucle frame a frame y la configuración del `VideoWriter`. A partir de ahí se han añadido las siguientes mejoras propias:

- Sustitución de `model.predict()` por `model.track()` para obtener IDs de seguimiento únicos por vehículo.
- Implementación de la lógica de línea virtual: se compara el centro vertical de cada caja con `LINEA_Y` y se usa un `set` de IDs ya contados para evitar duplicados.
- Visualización del conteo desglosado por categoría directamente en el fotograma.
- Progreso por consola cada 50 frames y tabla de resultados final.

### Prompts lanzados a la IA y análisis de las respuestas

**Prompt 1:** *"¿Cuál es la diferencia entre model.predict() y model.track() en Ultralytics YOLO?"*

> La IA explicó que `predict()` devuelve detecciones independientes por fotograma sin ningún tipo de seguimiento, mientras que `track()` asigna un ID persistente a cada objeto a lo largo del tiempo usando algoritmos como ByteTrack o BoT-SORT. Esto es fundamental para el conteo, ya que sin IDs únicos un mismo coche sería contado en cada fotograma. Se aplicó cambiando toda la lógica de detección a `model.track(persist=True)`.

**Prompt 2:** *"¿Cómo accedo a los IDs de tracking desde el objeto results de YOLO?"*

> La IA indicó que los IDs están en `results[0].boxes.id`, que puede ser `None` si ningún objeto fue rastreado en ese fotograma (por eso se añadió la comprobación `if results[0].boxes.id is not None`). Se aplicó directamente en el bucle de conteo.

**Prompt 3:** *"¿Cómo dibujo una línea horizontal fija en todos los fotogramas de un vídeo con OpenCV?"*

> La IA mostró el uso de `cv2.line(frame, pt1, pt2, color, thickness)`, aclarando que las coordenadas son `(x, y)` y que hay que llamarlo sobre el fotograma antes de escribirlo con `writer.write()`. Se aplicó tal cual en el código, añadiendo también un `cv2.putText` con la etiqueta.